In [ ]:
# !pip install tensorflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Load hourly Bitcoin price data
df_price_btc = pd.read_csv("data/btc1h_usdt.csv", 
                            parse_dates=["open_time"], 
                            index_col=["open_time"])

# Load daily hash rate data
df_hash_rate = pd.read_csv("data/hash_rate.csv", 
                            parse_dates=["date"], 
                            index_col=["date"])

# Check the shape and structure
print(df_price_btc.shape)
print(df_hash_rate.shape)
display(df_price_btc.head())
display(df_hash_rate.head())

# --- Step 1: Aggregate hourly BTC price to daily data ---
df_price_daily = df_price_btc.resample('1D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})

# Drop days with missing essential data
df_price_daily.dropna(subset=['open', 'high', 'low', 'close'], inplace=True)

# --- Step 2: Normalize datetime index for both datasets ---
df_price_daily.index = df_price_daily.index.normalize()
df_hash_rate.index = df_hash_rate.index.normalize()

# --- Step 3: Merge price and hash rate on date ---
df_merged = df_price_daily.merge(df_hash_rate, left_index=True, right_index=True, how='inner')

# --- Step 4: Preview merged data ---
display(df_merged.head())
